In this notebook I explored HF API to load BERT models.

In [3]:
from huggingface_hub import list_models, HfApi

# Method 2: Using HfApi client (useful if you need a custom token/endpoint)
api = HfApi()  # optional: use if private models need auth
models = api.list_models(author="Cyber-ThreaD")

for model in models:
    # print(model.modelId, model.downloads, model.tags)
    print(model.cardData)


None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None


In [26]:
models = api.list_models(author="Cyber-ThreaD")
models_by_model = {}
for model in models:
    # print(model.tags)
    name_data = model.modelId.split("/")[-1] 
    name = name_data.split("-")[0]
    if name not in models_by_model:
        models_by_model[name] = []
    models_by_model[name].append(name_data)
models_by_model

{'SecureBERT': ['SecureBERT-APTNER',
  'SecureBERT-DNRTI',
  'SecureBERT-CyNER',
  'SecureBERT-AttackER'],
 'CyBERT': ['CyBERT-CyNER',
  'CyBERT-DNRTI',
  'CyBERT-APTNER',
  'CyBERT-AttackER'],
 'SecBERT': ['SecBERT-APTNER',
  'SecBERT-DNRTI',
  'SecBERT-CyNER',
  'SecBERT-AttackER'],
 'DeBERTa': ['DeBERTa-CyNER',
  'DeBERTa-APTNER',
  'DeBERTa-DNRTI',
  'DeBERTa-v3-AttackER'],
 'RoBERTa': ['RoBERTa-DNRTI',
  'RoBERTa-CyNER',
  'RoBERTa-APTNER',
  'RoBERTa-AttackER']}

In [48]:
from dotenv import load_dotenv
from transformers import AutoModelForTokenClassification
import torch
load_dotenv()
# use mps if available, otherwise use cpu
# tor

model = AutoModelForTokenClassification.from_pretrained("Cyber-ThreaD/SecureBERT-DNRTI", use_auth_token=True)

TypeError: RobertaForTokenClassification.__init__() got an unexpected keyword argument 'use_auth_token'

In [46]:
# CYNER
# inspect model's classifier head
print(model.classifier)
# inspect predicted classes labels
print(set(v.split('-')[-1] for k,v in model.config.id2label.items()))

Linear(in_features=768, out_features=11, bias=True)
{'Organization', 'Indicator', 'System', 'Vulnerability', 'O', 'Malware'}


In [ ]:
# inspect model's classifier head
print(model.classifier)
# inspect predicted classes labels
print(set(v.split('-')[-1] for k,v in model.config.id2label.items()))

Linear(in_features=768, out_features=11, bias=True)
{'Organization', 'Indicator', 'System', 'Vulnerability', 'O', 'Malware'}


In [ ]:
# inspect model's classifier head
print(model.classifier)
# inspect predicted classes labels
print(set(v.split('-')[-1] for k,v in model.config.id2label.items()))

Linear(in_features=768, out_features=11, bias=True)
{'Organization', 'Indicator', 'System', 'Vulnerability', 'O', 'Malware'}


In [ ]:
# inspect model's classifier head
print(model.classifier)
# inspect predicted classes labels
print(set(v.split('-')[-1] for k,v in model.config.id2label.items()))

Linear(in_features=768, out_features=11, bias=True)
{'Organization', 'Indicator', 'System', 'Vulnerability', 'O', 'Malware'}


In [ ]:
# plan:
# 1. load load: securebert for dnrti
# 2. load unified cyner dataset
# 3. run the model on the dataset
# 4. perform late-merging of the predictions per #unify_labels_dnrti function description
# 5. compute the metrics: using #span_f1 function


TypeError: cannot unpack non-iterable int object

In [2]:
# 1. Load SecureBERT fine-tuned on DNRTI
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

from nlp_cyber_ner.config import PROCESSED_DATA_DIR, load_dotenv
from nlp_cyber_ner.dataset import read_iob2_file
from nlp_cyber_ner.span_f1 import span_f1

load_dotenv()

MODEL_ID = "Cyber-ThreaD/SecureBERT-DNRTI"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForTokenClassification.from_pretrained(MODEL_ID, token=True)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
model = model.to(device).eval()
id2label = model.config.id2label
print("DNRTI raw labels:", sorted({v.split("-", 1)[-1] for v in id2label.values()}))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24399.75it/s]

DNRTI raw labels: ['Area', 'Exp', 'Features', 'HackOrg', 'Idus', 'O', 'OffAct', 'Org', 'Purp', 'SamFile', 'SecTeam', 'Time', 'Tool', 'Way']


In [3]:
# 2. Load the (cleaned/unified) CyNER test set. Labels already in unified space
#    (Organization / System / Vulnerability / Malware); Indicator was dropped.
cyner_test = read_iob2_file(PROCESSED_DATA_DIR / "cyner" / "test.unified")
print(f"{len(cyner_test)} sentences")
print("gold label set:", sorted({t for _, tags in cyner_test for t in tags}))

748 sentences
gold label set: ['B-Malware', 'B-Organization', 'B-System', 'B-Vulnerability', 'I-Malware', 'I-Organization', 'I-System', 'I-Vulnerability', 'O']


In [4]:
# 3. Run the DNRTI model on the CyNER tokens.
#    Words are pre-tokenized -> use is_split_into_words and take the FIRST
#    subword's prediction per word (see cross_dataset_model.py for the eval shape:
#    span_f1 expects one predicted tag per gold token, per sentence).
@torch.no_grad()
def predict_dnrti(words: list[str]) -> list[str]:
    enc = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )
    logits = model(**{k: v.to(device) for k, v in enc.items()}).logits
    pred_ids = logits.argmax(-1)[0].tolist()

    tags, prev = [], None
    for wid, pid in zip(enc.word_ids(0), pred_ids):
        if wid is None or wid == prev:  # special token or non-first subword
            continue
        tags.append(id2label[pid])
        prev = wid
    # if truncation dropped trailing words, pad so lengths line up with gold
    tags += ["O"] * (len(words) - len(tags))
    return tags


raw_preds = [predict_dnrti(words) for words, _ in cyner_test]
print("example raw DNRTI prediction:", raw_preds[0][:15])

example raw DNRTI prediction: ['O', 'O', 'O', 'O', 'O', 'O', 'B-Area', 'O']


In [5]:
# 4. Late-merge the DNRTI predictions into the unified label space,
#    following the mapping in unify_labels_dnrti (dataset.py):
#      HackOrg, SecTeam, Org -> Organization
#      Tool                  -> System
#      Exp, Way              -> Vulnerability
#      SamFile               -> Malware
#      everything else       -> O
#    The B-/I- prefix is preserved.
DNRTI_TO_UNIFIED = {
    "HackOrg": "Organization",
    "SecTeam": "Organization",
    "Org": "Organization",
    "Tool": "System",
    "Exp": "Vulnerability",
    "Way": "Vulnerability",
    "SamFile": "Malware",
}


def unify_tag(tag: str) -> str:
    if tag == "O":
        return "O"
    prefix, label = tag.split("-", 1)
    mapped = DNRTI_TO_UNIFIED.get(label)
    return f"{prefix}-{mapped}" if mapped else "O"


pred_unified = [[unify_tag(t) for t in sent] for sent in raw_preds]
print("example unified prediction:", pred_unified[0][:15])

example unified prediction: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [6]:
# 5. Compute span-F1 (strict / unlabeled / loose) of the DNRTI model
#    evaluated cross-dataset on unified CyNER.
gold = [tags for _, tags in cyner_test]
assert all(len(g) == len(p) for g, p in zip(gold, pred_unified)), "length mismatch"

metrics = span_f1(gold, pred_unified)
metrics

recall:    0.19809825673534073
precision: 0.16512549537648613
slot-f1:   0.1801152737752161

unlabeled
ul_recall:    0.410459587955626
ul_precision: 0.34214002642007924
ul_slot-f1:   0.3731988472622478

loose (partial overlap with same label)
l_recall:    0.2630744849445325
l_precision: 0.20607661822985468
l_slot-f1:   0.23111317370079162


{'recall': 0.19809825673534073,
 'precision': 0.16512549537648613,
 'slot-f1': 0.1801152737752161,
 'ul_recall': 0.410459587955626,
 'ul_precision': 0.34214002642007924,
 'ul_slot-f1': 0.3731988472622478,
 'l_recall': 0.2630744849445325,
 'l_precision': 0.20607661822985468,
 'l_slot-f1': 0.23111317370079162}